# Dense Retrieval: Semantic Search with Embeddings

**Dense retrieval** matches queries and documents by *meaning* instead of keywords. A bi-encoder (`sentence-transformers/all-MiniLM-L6-v2`) maps every text to a normalized 384-dimensional vector:

```
"total revenues in 2024"  -> [0.12, -0.03, 0.45, ...]
"annual income last year" -> [0.10, -0.01, 0.47, ...]
```

Semantically similar texts land close together, so retrieval becomes a nearest-neighbor search. Because the vectors are normalized, cosine similarity is just a dot product.

In this notebook we embed sample texts, build a FAISS-backed index over the 10-K chunks, and see dense retrieval handle paraphrases that BM25 misses.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from rag.data_ingestion import load_documents, chunk_documents
from rag.embeddings import embed_texts
from rag.retrievers import DenseRetriever

PDF_PATH = ROOT / "data" / "google_10K.pdf"

documents = load_documents(PDF_PATH)
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")

Loaded 107 pages -> 433 chunks


## Embeddings up close

`embed_texts` returns a `(n, 384)` float32 array with unit-norm rows.

In [2]:
texts = [
    "Total revenues in fiscal year 2024",
    "Annual income last year was high",
    "The weather is sunny today",
]

embeddings = embed_texts(texts)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Norm of first embedding: {np.linalg.norm(embeddings[0]):.4f}")
print(f"\nFirst 10 dimensions of embedding 0:\n{embeddings[0][:10]}")

Shape: (3, 384)
Dtype: float32
Norm of first embedding: 1.0000

First 10 dimensions of embedding 0:
[ 0.01725245 -0.0145872  -0.01352398 -0.07739183 -0.07238472  0.0109903
 -0.02382093  0.0309123  -0.02695938  0.02023664]


## Cosine similarity

For normalized vectors, `embeddings @ embeddings.T` gives the full cosine-similarity matrix. The two revenue-related sentences should be much closer to each other than to the weather sentence.

In [3]:
similarity = embeddings @ embeddings.T

print("Similarity matrix:")
print(np.round(similarity, 3))

print(f"\nrevenue vs income:  {similarity[0, 1]:.3f}  (related)")
print(f"revenue vs weather: {similarity[0, 2]:.3f}  (unrelated)")
print(f"income  vs weather: {similarity[1, 2]:.3f}  (unrelated)")

Similarity matrix:
[[ 1.     0.49  -0.003]
 [ 0.49   1.     0.105]
 [-0.003  0.105  1.   ]]

revenue vs income:  0.490  (related)
revenue vs weather: -0.003  (unrelated)
income  vs weather: 0.105  (unrelated)


## Building the index

`DenseRetriever.add_documents` embeds every chunk once and builds a FAISS inner-product index (falling back to brute-force NumPy if FAISS is unavailable).

In [4]:
retriever = DenseRetriever()
retriever.add_documents(chunks)

print(f"Indexed {len(retriever.documents)} chunks")
print(f"Embedding matrix: {retriever.embeddings.shape}")
print(f"Using FAISS index: {retriever.index is not None}")

Indexed 433 chunks
Embedding matrix: (433, 384)
Using FAISS index: True


## Semantic retrieval

This query shares almost no vocabulary with the filing — BM25 would struggle, but the embedding still lands near the revenue discussion.

In [5]:
query = "How much money did the company make last year?"

results = retriever.retrieve(query, top_k=5)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    preview = doc.page_content[:150].replace("\n", " ")
    print(f"{i}. [page {doc.metadata['page']}] {preview}...\n")

Query: How much money did the company make last year?

1. [page 41] e in a valuation-based compensation charge related to Waymo. Sales and Marketing The following table presents sales and marketing expenses (in million...

2. [page 41]  revenues were substantially consistent from 2024 to 2025. The increase in other cost of revenues from 2024 to 2025 was primarily due to increases in ...

3. [page 39] uarter for Waymo, primarily reflected in research and development expenses, based on estimated stock valuation. In February 2026, Waymo announced an i...

4. [page 38] , 2024 2025 $ Change % Change Consolidated revenues $ 350,018  $ 402,836  $ 52,818  15 % Cost of revenues $ 146,306  $ 162,535  $ 16,229  11 % Operati...

5. [page 39] Table of Contents Alphabet Inc. • In 2025, we entered into definitive agreements to acquire Wiz, a leading cloud security platform, for $32.0 billion,...



## Scores against the whole corpus

Embedding the query and taking dot products with the stored chunk embeddings gives a similarity score for every chunk.

In [6]:
query = "total revenues 2024"
query_emb = embed_texts([query])[0]

similarities = retriever.embeddings @ query_emb
top_indices = np.argsort(similarities)[::-1][:10]

print(f"Query: {query}\n")
print(f"{'Rank':<6} {'Similarity':<12} {'Page':<6} Preview")
print("-" * 70)
for rank, idx in enumerate(top_indices, start=1):
    doc = retriever.documents[idx]
    preview = doc.page_content[:40].replace("\n", " ")
    print(f"{rank:<6} {similarities[idx]:<12.4f} {doc.metadata['page']:<6} {preview}...")

Query: total revenues 2024

Rank   Similarity   Page   Preview
----------------------------------------------------------------------
1      0.5561       41      revenues were substantially consistent ...
2      0.5408       57     December 31, 2025 12,088  $ 93,126  $ (1...
3      0.5232       39     he year ended December 31, 2025. • As of...
4      0.5200       38     , 2024 2025 $ Change % Change Consolidat...
5      0.5140       66     or less and cancellable contracts. Defer...
6      0.5136       80     s of December 31, 2025 $ (2,558) $ 678  ...
7      0.4984       41     Table of Contents Alphabet Inc. Cost of ...
8      0.4964       40     One. Google Cloud Google Cloud revenues ...
9      0.4961       41     e in a valuation-based compensation char...
10     0.4792       32     ond Advertising: Revenues from cloud, co...


## Robustness to phrasing

Different phrasings of the same question produce similar top scores — the strength dense retrieval adds over BM25.

In [7]:
variants = [
    "total revenues 2024",
    "How much did they earn?",
    "Annual income last year",
    "yearly sales in 2024",
]

print(f"{'Query':<28} {'Top score':<12} {'2nd':<10} {'3rd':<10}")
print("-" * 62)
for query in variants:
    sims = retriever.embeddings @ embed_texts([query])[0]
    top3 = sorted(sims, reverse=True)[:3]
    print(f"{query:<28} {top3[0]:<12.4f} {top3[1]:<10.4f} {top3[2]:<10.4f}")

Query                        Top score    2nd        3rd       
--------------------------------------------------------------
total revenues 2024          0.5561       0.5408     0.5232    
How much did they earn?      0.4226       0.4088     0.4063    
Annual income last year      0.4392       0.4282     0.4111    
yearly sales in 2024         0.5239       0.4633     0.4163    


## Query latency

Chunk embeddings are computed once at indexing time; each search only embeds the query and does one nearest-neighbor lookup.

In [8]:
import time

queries = ["total revenues", "operating income", "net income", "total assets", "stockholders equity"]

print(f"{'Query':<28} {'Time (ms)':>10}")
print("-" * 40)
for query in queries:
    start = time.time()
    retriever.retrieve(query, top_k=5)
    print(f"{query:<28} {(time.time() - start) * 1000:>10.1f}")

Query                         Time (ms)
----------------------------------------
total revenues                      5.5
operating income                    5.4
net income                          6.3
total assets                        4.7
stockholders equity                 7.0


## Persistence

Re-embedding 400+ chunks on every run is wasteful. `save`/`load` persist the documents, the embedding matrix, and the FAISS index.

In [9]:
index_path = ROOT / "tmp_dense_index"
retriever.save(index_path)

print(f"Saved to {index_path}:")
for f in sorted(index_path.glob("*")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.0f} KB")

Saved to /home/nick/Sec-Rag/tmp_dense_index:
  documents.pkl: 391 KB
  embeddings.npy: 650 KB
  index.faiss: 650 KB


In [10]:
loaded = DenseRetriever()
loaded.load(index_path)

results = loaded.retrieve("total revenues", top_k=3)
print(f"Loaded {len(loaded.documents)} documents; retrieved {len(results)} results")

Loaded 433 documents; retrieved 3 results


In [11]:
import shutil

shutil.rmtree(index_path)
print(f"Cleaned up {index_path}")

Cleaned up /home/nick/Sec-Rag/tmp_dense_index


## Takeaways

- Dense retrieval matches by meaning, so paraphrases and conversational questions work.
- Indexing cost is paid once; queries are fast.
- It can miss exact identifiers (tickers, precise line items) that BM25 nails.

`03_hybrid.ipynb` combines both approaches to get the best of each.